In [ ]:
pip install -U pyabsa

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.2/574.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.4 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=88cac527f054d21767591022df87f6c4d6fd123d79f14b25134110413ba13565
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
# Lệnh này sẽ liệt kê nội dung trong Drive của bạn sau khi Mount
!ls /content/drive/MyDrive/
# Dùng tên lối tắt bạn đã tạo
!ls /content/drive/MyDrive/MockAIProject_062025/model_absa

 123-AfterTheRain-6151191.mp3
 200_200_232.gdoc
 20250606_092707.jpg
'20250606_095201 (1).jpg'
 20250606_095201.jpg
 20250606_095710.jpg
 AiWoKometeUmi-AoiTeshima-2868900.mp3
 anh
 AnoYumeWoNazotte-AyaseYOASOBI-6236943.mp3
 backupgenshn
'Bản sao của Lab5-4.tinh.pkt'
'Bản trình bày không có tiêu đề.gslides'
 BeBrave-ExileAtsushiAi-4107938_hq.mp3
 BoccaDellaVerita-Sou-6272109.mp3
'Class diagram.drawio'
 class.drawio
 Classroom
'Colab Notebooks'
 CynicalNightPlan-Sou-6262166.mp3
 DACN2
'deeplearning cousera'
'đồ án cơ sở 1'
'đồ án cơ sở 1.rar'
 doimatkhau.drawio
 Ember.Knights.v1.2.1_LinkNeverDie.Com.rar
 Expert.m4a
 Gehenna-WotakuHatsuneMiku-6253937.mp3
'Google AI Studio'
'HanaNiBourei-Yorushika-6264556 (1).mp3'
 HanaNiBourei-Yorushika-6264556.mp3
 HarehareYa-Sou-5492450.mp3
 HaruHisagi-Yorushika-6288857.mp3
'HatenaGundamBuildDiversReRise2ndSeasonOpening-PenguinResearch-6262832 (1).mp3'
 HatenaGundamBuildDiversReRise2ndSeasonOpening-PenguinResearch-6262832.mp3
 Hazy

In [ ]:
import os
from pathlib import Path

# Đường dẫn đến file cần kiểm tra
file_path = '/content/drive/MyDrive/model_absa'

## Cách 1: Sử dụng os.path.exists()
print("--- Cách 1: Sử dụng os.path.exists() ---")
if os.path.exists(file_path):
    print(f"✅ File '{file_path}' Tồn tại.")
else:
    print(f"❌ File '{file_path}' KHÔNG tồn tại.")

print("\n")

## Cách 2: Sử dụng pathlib.Path.exists()
print("--- Cách 2: Sử dụng pathlib.Path.exists() ---")
path_obj = Path(file_path)
if path_obj.exists():
    print(f"✅ File '{file_path}' Tồn tại.")
else:
    print(f"❌ File '{file_path}' KHÔNG tồn tại.")

--- Cách 1: Sử dụng os.path.exists() ---
✅ File '/content/drive/MyDrive/model_absa' Tồn tại.


--- Cách 2: Sử dụng pathlib.Path.exists() ---
✅ File '/content/drive/MyDrive/model_absa' Tồn tại.


In [ ]:
!mkdir final_model_safe

In [ ]:
from pyabsa import AspectPolarityClassification as APC
import os
import glob
from tqdm.auto import tqdm
# Sử dụng MODEL_DRIVE_PATH đã định nghĩa
MODEL_DRIVE_PATH = '/content/drive/MyDrive/model_absa'

# -----------------------------------------------------------
# I. HÀM TẢI MODEL ĐÃ SỬA (CHỈ LOAD TỪ DRIVE PATH)
# -----------------------------------------------------------
def initialize_classifier():
    """Tải model chỉ từ đường dẫn cố định đã mount trong Drive."""

    # Kiểm tra tệp trọng số quan trọng nhất
    state_dict_path = os.path.join(MODEL_DRIVE_PATH, 'lcf_bert.state_dict')

    if os.path.exists(state_dict_path):
        print(f"🔄 Đang tải model từ đường dẫn Drive: {MODEL_DRIVE_PATH}")
        # PyABSA sẽ tải model từ đường dẫn tuyệt đối đã mount
        return APC.SentimentClassifier(checkpoint=MODEL_DRIVE_PATH)

    # Báo lỗi rõ ràng nếu không tìm thấy tệp trọng số
    raise RuntimeError(f"❌ LỖI: Không tìm thấy tệp trọng số '{state_dict_path}'. Vui lòng kiểm tra lại đường dẫn MODEL_DRIVE_PATH và chắc chắn đã Mount Drive.")

# 11 nhãn Aspect bạn đã cung cấp:
ASPECT_LABELS = [
    "connectivity", "design_features", "general", "miscellaneous",
    "operation_performance", "portability", "price", "prices",
    "quality", "style_options", "usability"
]

# Các Scenario phức tạp cần kiểm tra (ĐÃ XÓA DẤU **)
SCENARIOS = [
    {
        "sentence": "The laptop overall quality is excellent, but the price is far too high for this little change.",
        "expected_results": {
            "quality": "Positive",
            "price": "Negative",
        },
        "description": "Kiểm tra xung đột giữa Chất lượng và Giá cả."
    },
    {
        "sentence": "The Laptopdesign is amazing, but when loading apps, the performance is awful and crashes often.",
        "expected_results": {
            "design_features": "Positive",
            "operation_performance": "Negative",
            "usability": "Negative"
        },
        "description": "Kiểm tra xung đột giữa Thiết kế (Design) và Hiệu suất (Performance)."
    },
    {
        "sentence": "Laptop is lightweight and easy to carry, but the Wi-Fi keeps disconnecting every hour.",
        "expected_results": {
            "portability": "Positive",
            "connectivity": "Negative",
        },
        "description": "Kiểm tra xung đột giữa Tính di động và Kết nối."
    },
    {"sentence": "Laptop is lightweight and easy to carry, but the meal is awful.",
    }
]

# --- THỰC THI VÒNG LẶP TEST ---
try:
    classifier = initialize_classifier()

    for idx, scenario in enumerate(SCENARIOS):
        description = scenario.get('description', f"Kịch bản {idx+1} (Phân tích cảm xúc chung)")
        print(f"\n--- SCENARIO {idx+1}: {description} ---")

        # Xóa dấu '**' khỏi câu trước khi in và chạy model
        clean_sentence = scenario['sentence'].replace('**', '')

        if 'expected_results' in scenario:
            aspects_to_run = list(scenario['expected_results'].keys())
            results = []

            for aspect in aspects_to_run:
                # Dùng câu đã được làm sạch cho input model
                input_text = f"{clean_sentence} Aspect: {aspect}"
                result = classifier.predict(text=input_text, print_result=False)

                results.append({
                    'Aspect': aspect,
                    'Polarity': result['sentiment'][0],
                    'Confidence': max(result['probs'][0]),
                    'Expected': scenario['expected_results'][aspect]
                })

            # --- IN KẾT QUẢ CHO SCENARIO CÓ EXPECTED ---
            print(f"Câu: {clean_sentence}")
            print("{:<20} {:<10} {:<10} {:<10}".format("SENTIMENT FOR", "EXPECTED", "PREDICTED", "CONFIDENCE"))
            print("-" * 50)

            for item in results:
                # Vẫn in ra item['Aspect'] để người dùng biết cảm xúc này dành cho khía cạnh nào
                print("{:<20} {:<10} {:<10} {:<10.4f} ".format(
                    item['Aspect'],
                    item['Expected'],
                    item['Polarity'],
                    item['Confidence'])
                )
            print("-" * 50)

        else:
            # --- IN KẾT QUẢ CHO SCENARIO KHÔNG CÓ EXPECTED (General Sentiment) ---
            print(f"Câu: {clean_sentence}")

            result_general = classifier.predict(text=clean_sentence, print_result=False)

            predicted_sentiment = result_general['sentiment'][0]
            confidence = max(result_general['probs'][0])

            print("\n--- KẾT QUẢ DỰ ĐOÁN CHUNG ---")
            print("{:<25} {:<15}".format("Sentiment", "Confidence"))
            print("-" * 40)
            print("{:<25} {:<15.4f}".format(predicted_sentiment, confidence))
            print("-" * 40)

    # -------------------------------------------------------------
    # PHẦN MINH HỌA (GENERAL SENTIMENT - KHÔNG CÓ ASPECT HOÀN TOÀN)
    # -------------------------------------------------------------

    # Dùng câu đã được làm sạch từ SCENARIOS[0]
    example_sentence_clean = SCENARIOS[0]['sentence'].replace('**', '')


    print(f"📌 Câu đầu vào: {example_sentence_clean}")

    result_no_aspect = classifier.predict(text=example_sentence_clean, print_result=False)

    predicted_sentiment = result_no_aspect['sentiment'][0]
    confidence = max(result_no_aspect['probs'][0])

    print("\n--- KẾT QUẢ DỰ ĐOÁN CHUNG ---")

    print("\n{:<25} {:<15}".format("TRƯỜNG THÔNG TIN", "KẾT QUẢ DỰ ĐOÁN"))
    print("-" * 40)

    print("{:<25} {:<15}".format("Sentiment", predicted_sentiment))
    print("{:<25} {:<15.4f}".format("Confidence", confidence))
    print("-" * 40)

    print("\n💡 **LƯU Ý QUAN TRỌNG:**")


except RuntimeError as e:
    print(f"\n❌ KHÔNG THỂ CHẠY TEST: {e}")

🔄 Đang tải model từ đường dẫn Drive: /content/drive/MyDrive/model_absa
[2025-12-06 02:17:51] (2.4.2) Load sentiment classifier from /content/drive/MyDrive/model_absa
[2025-12-06 02:17:51] (2.4.2) config: drive/MyDrive/model_absa/lcf_bert.config
[2025-12-06 02:17:51] (2.4.2) state_dict: drive/MyDrive/model_absa/lcf_bert.state_dict
[2025-12-06 02:17:51] (2.4.2) model: None
[2025-12-06 02:17:51] (2.4.2) tokenizer: drive/MyDrive/model_absa/lcf_bert.tokenizer
[2025-12-06 02:17:51] (2.4.2) Set Model Device: cuda:0
[2025-12-06 02:17:51] (2.4.2) Device Name: Tesla T4

--- SCENARIO 1: Kiểm tra xung đột giữa Chất lượng và Giá cả. ---
[2025-12-06 02:17:57] (2.4.2) Warning: reference sentiment does not exist or its number 0 is not equal to aspect number 1, text:  [B-ASP]Global Sentiment[E-ASP] The laptop overall quality is excellent, but the price is far too high for this little change. Aspect: quality
[2025-12-06 02:17:57] (2.4.2) Warning: reference sentiment does not exist or its number 0 is not

In [ ]:
from pyabsa import AspectPolarityClassification as APC
import os
from tqdm.auto import tqdm

# -----------------------------------------------------------
# I. TẢI MODEL TỪ GOOGLE DRIVE
# -----------------------------------------------------------
MODEL_DRIVE_PATH = '/content/drive/MyDrive/model_absa'

def initialize_classifier():
    """Tải model từ đường dẫn trong Drive."""
    state_dict_path = os.path.join(MODEL_DRIVE_PATH, 'lcf_bert.state_dict')

    if os.path.exists(state_dict_path):
        print(f"🔄 Đang tải model từ Drive: {MODEL_DRIVE_PATH}")
        return APC.SentimentClassifier(checkpoint=MODEL_DRIVE_PATH)

    raise RuntimeError(f"❌ Không tìm thấy trọng số '{state_dict_path}'")


# -----------------------------------------------------------
# II. 11 NHÃN ASPECT CỦA MÔ HÌNH
# -----------------------------------------------------------
ASPECT_LABELS = [
    "connectivity", "design_features", "general", "miscellaneous",
    "operation_performance", "portability", "price", "prices",
    "quality", "style_options", "usability"
]


# -----------------------------------------------------------
# III. SCENARIOS (Restaurant + Mix Restaurant-Laptop)
# -----------------------------------------------------------
SCENARIOS = [

    # ------------------------- MIX DOMAIN -------------------------
    {
        "sentence": "The laptop’s performance was smooth, but the restaurant where I tested it had extremely slow Wi-Fi and inconsistent network service.",
        "expected_results": {
            "operation_performance": "Positive",
            "connectivity": "Negative"
        },
        "description": "Mix: Laptop hiệu năng tốt nhưng Wi-Fi nhà hàng rất tệ."
    },

    {
        "sentence": "I enjoyed how lightweight the laptop was, but the pasta I ordered at the restaurant tasted bland and undercooked.",
        "expected_results": {
            "portability": "Positive",
            "quality": "Negative"
        },
        "description": "Mix: Laptop nhẹ nhưng món ăn dở."
    },

    {
        "sentence": "The laptop keyboard provided an excellent typing experience, yet the restaurant service during lunch was painfully slow and unorganized.",
        "expected_results": {
            "usability": "Positive",
            "operation_performance": "Negative"
        },
        "description": "Mix: Laptop dùng thích nhưng phục vụ nhà hàng chậm."
    },

    # -----------------------------------------------------
    # ---------------------- RESTAURANT ONLY ----------------------
    # -----------------------------------------------------
    {
        "sentence": "The grilled chicken was flavorful and perfectly cooked, but the restaurant lighting was so dim that it felt uncomfortable.",
        "expected_results": {
            "quality": "Positive",
            "design_features": "Negative"
        },
        "description": "Restaurant: Món ăn ngon nhưng ánh sáng tệ."
    },

    {
        "sentence": "The staff were polite and friendly, but the waiting time for dishes was extremely long and frustrating.",
        "expected_results": {
            "usability": "Positive",
            "operation_performance": "Negative"
        },
        "description": "Restaurant: Nhân viên tốt nhưng đợi món lâu."
    },

    {
        "sentence": "The dessert was outstanding and the pricing was surprisingly reasonable for such a high-quality restaurant.",
        "expected_results": {
            "quality": "Positive",
            "price": "Positive"
        },
        "description": "Restaurant: Tráng miệng ngon, giá tốt."
    },

    {
        "sentence": "The soup was too salty and the seating arrangement felt cramped and uncomfortable.",
        "expected_results": {
            "quality": "Negative",
            "design_features": "Negative"
        },
        "description": "Restaurant: Món ăn mặn, bố trí bàn ghế xấu."
    },

    # ---------------------------- GENERAL SENTIMENT ----------------------------
    {
        "sentence": "The restaurant ambiance looked beautiful at first, but overall the dining experience ended up being disappointing.",
        # Không có expected_results → model sẽ chạy sentiment tổng
    }
]


# -----------------------------------------------------------
# IV. CHẠY KIỂM TRA TỪNG SCENARIO
# -----------------------------------------------------------
try:
    classifier = initialize_classifier()

    for idx, scenario in enumerate(SCENARIOS):
        desc = scenario.get("description", f"Kịch bản {idx+1} (General sentiment)")
        print(f"\n--- SCENARIO {idx+1}: {desc} ---")

        clean_sentence = scenario["sentence"]

        # -------------------------------------------------------
        # TRƯỜNG HỢP CÓ EXPECTED (ASPECT-BASED)
        # -------------------------------------------------------
        if "expected_results" in scenario:
            aspects = list(scenario["expected_results"].keys())
            results = []

            for aspect in aspects:
                model_input = f"{clean_sentence} Aspect: {aspect}"
                out = classifier.predict(text=model_input, print_result=False)

                results.append({
                    "Aspect": aspect,
                    "Expected": scenario["expected_results"][aspect],
                    "Predicted": out["sentiment"][0],
                    "Confidence": max(out["probs"][0])
                })

            # In bảng kết quả
            print(f"Câu: {clean_sentence}")
            print("{:<22} {:<12} {:<12} {:<10}".format("ASPECT", "EXPECTED", "PREDICTED", "CONFIDENCE"))
            print("-" * 60)

            for r in results:
                print("{:<22} {:<12} {:<12} {:.4f}".format(
                    r["Aspect"], r["Expected"], r["Predicted"], r["Confidence"]
                ))

            print("-" * 60)

        # -------------------------------------------------------
        # TRƯỜNG HỢP KHÔNG EXPECTED → SENTIMENT CHUNG
        # -------------------------------------------------------
        else:
            print(f"Câu: {clean_sentence}")
            result = classifier.predict(text=clean_sentence, print_result=False)

            print("\n--- KẾT QUẢ SENTIMENT CHUNG ---")
            print("{:<20} {:<15}".format("Sentiment", "Confidence"))
            print("-" * 40)
            print("{:<20} {:.4f}".format(result["sentiment"][0], max(result["probs"][0])))
            print("-" * 40)

except RuntimeError as e:
    print(f"❌ KHÔNG THỂ CHẠY TEST: {e}")


🔄 Đang tải model từ Drive: /content/drive/MyDrive/model_absa
[2025-12-06 02:57:51] (2.4.2) Load sentiment classifier from /content/drive/MyDrive/model_absa
[2025-12-06 02:57:52] (2.4.2) config: drive/MyDrive/model_absa/lcf_bert.config
[2025-12-06 02:57:52] (2.4.2) state_dict: drive/MyDrive/model_absa/lcf_bert.state_dict
[2025-12-06 02:57:52] (2.4.2) model: None
[2025-12-06 02:57:52] (2.4.2) tokenizer: drive/MyDrive/model_absa/lcf_bert.tokenizer
[2025-12-06 02:57:52] (2.4.2) Set Model Device: cuda:0
[2025-12-06 02:57:52] (2.4.2) Device Name: Tesla T4

--- SCENARIO 1: Mix: Laptop hiệu năng tốt nhưng Wi-Fi nhà hàng rất tệ. ---
[2025-12-06 02:57:57] (2.4.2) Warning: reference sentiment does not exist or its number 0 is not equal to aspect number 1, text:  [B-ASP]Global Sentiment[E-ASP] The laptop’s performance was smooth, but the restaurant where I tested it had extremely slow Wi-Fi and inconsistent network service. Aspect: operation_performance
[2025-12-06 02:57:57] (2.4.2) Warning: refer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pyabsa import AspectPolarityClassification as APC
import os
import glob
from tqdm.auto import tqdm

# Sử dụng MODEL_DRIVE_PATH đã định nghĩa (Giả định đã được mount)
MODEL_DRIVE_PATH = '/content/drive/MyDrive/model_absa'

# -----------------------------------------------------------
# I. HÀM TẢI MODEL (Giữ nguyên)
# -----------------------------------------------------------
def initialize_classifier():
    """Tải model chỉ từ đường dẫn cố định đã mount trong Drive."""

    state_dict_path = os.path.join(MODEL_DRIVE_PATH, 'lcf_bert.state_dict')

    if os.path.exists(state_dict_path):
        print(f"🔄 Đang tải model từ đường dẫn Drive: {MODEL_DRIVE_PATH}")
        return APC.SentimentClassifier(checkpoint=MODEL_DRIVE_PATH)

    raise RuntimeError(f"❌ LỖI: Không tìm thấy tệp trọng số '{state_dict_path}'. Vui lòng kiểm tra lại đường dẫn MODEL_DRIVE_PATH.")


# -----------------------------------------------------------
# II. VÍ DỤ PHÂN TÍCH ĐA KHÍA CẠNH
# -----------------------------------------------------------

# 11 nhãn Aspect bạn đã cung cấp:
ASPECT_LABELS = [
    "connectivity", "design_features", "general", "miscellaneous",
    "operation_performance", "portability", "price", "prices",
    "quality", "style_options", "usability"
]

# Câu văn chứa nhiều xung đột cảm xúc
COMPLEX_SENTENCE = "The Laptop  is sleek and beautiful, and its  is great for travel. However, the cost is too steep and  slows down when gaming."

# Các khía cạnh cần kiểm tra trong câu này (Aspects in the sentence)
ASPECTS_TO_CHECK = [
    "connectivity", "design_features", "general", "miscellaneous","operation_performance", "portability", "price", "prices","quality", "style_options", "usability"
]


try:
    classifier = initialize_classifier()

    print("\n" + "="*80)
    print("🔬 PHÂN TÍCH CẢM XÚC THEO KHÍA CẠNH (ABSA) CHO CÂU VÍ DỤ")
    print("="*80)
    print(f"Câu: {COMPLEX_SENTENCE.replace('**', '')}")

    results = []

    # --- THỰC THI DỰ ĐOÁN CHO TỪNG ASPECT ---
    for aspect in ASPECTS_TO_CHECK:
        # Tạo Input: Câu + Aspect
        input_text = f"{COMPLEX_SENTENCE.replace('**', '')} Aspect: {aspect}"

        # Chạy mô hình phân loại cảm xúc
        result = classifier.predict(text=input_text, print_result=False)

        results.append({
            'Aspect': aspect,
            'Polarity': result['sentiment'][0],
            'Confidence': max(result['probs'][0])
        })

    # --- IN KẾT QUẢ TỔNG HỢP ---
    print("\n--- KẾT QUẢ DỰ ĐOÁN CẢM XÚC THEO TỪNG KHÍA CẠNH ---")
    print("{:<25} {:<15} {:<15}".format("ASPECT", "POLARITY", "CONFIDENCE"))
    print("-" * 55)

    for item in results:
        print("{:<25} {:<15}  ".format(
            item['Aspect'],
            item['Polarity'],
            item['Confidence'])
        )
    print("-" * 55)

except RuntimeError as e:
    print(f"\n❌ KHÔNG THỂ CHẠY PHÂN TÍCH: {e}")
ASPECTS_TO_CHECK1 = [
    "design_features", "portability", "price", "operation_performance"
]


try:
    classifier = initialize_classifier()

    print("\n" + "="*80)
    print("🔬 PHÂN TÍCH CẢM XÚC THEO KHÍA CẠNH (ABSA) CHO CÂU VÍ DỤ")
    print("="*80)
    print(f"Câu: {COMPLEX_SENTENCE.replace('**', '')}")

    results = []

    # --- THỰC THI DỰ ĐOÁN CHO TỪNG ASPECT ---
    for aspect in ASPECTS_TO_CHECK1:
        # Tạo Input: Câu + Aspect
        input_text = f"{COMPLEX_SENTENCE.replace('**', '')} Aspect: {aspect}"

        # Chạy mô hình phân loại cảm xúc
        result = classifier.predict(text=input_text, print_result=False)

        results.append({
            'Aspect': aspect,
            'Polarity': result['sentiment'][0],
            'Confidence': max(result['probs'][0])
        })

    # --- IN KẾT QUẢ TỔNG HỢP ---
    print("\n--- KẾT QUẢ DỰ ĐOÁN CẢM XÚC THEO TỪNG KHÍA CẠNH ---")
    print("{:<25} {:<15} {:<15}".format("ASPECT", "POLARITY", "CONFIDENCE"))
    print("-" * 55)

    for item in results:
        print("{:<25} {:<15} ".format(
            item['Aspect'],
            item['Polarity'],
            )
        )
    print("-" * 55)

except RuntimeError as e:
    print(f"\n❌ KHÔNG THỂ CHẠY PHÂN TÍCH: {e}")

🔄 Đang tải model từ đường dẫn Drive: /content/drive/MyDrive/model_absa
[2025-12-06 02:47:11] (2.4.2) Load sentiment classifier from /content/drive/MyDrive/model_absa
[2025-12-06 02:47:11] (2.4.2) config: drive/MyDrive/model_absa/lcf_bert.config
[2025-12-06 02:47:11] (2.4.2) state_dict: drive/MyDrive/model_absa/lcf_bert.state_dict
[2025-12-06 02:47:11] (2.4.2) model: None
[2025-12-06 02:47:11] (2.4.2) tokenizer: drive/MyDrive/model_absa/lcf_bert.tokenizer
[2025-12-06 02:47:12] (2.4.2) Set Model Device: cuda:0
[2025-12-06 02:47:12] (2.4.2) Device Name: Tesla T4

🔬 PHÂN TÍCH CẢM XÚC THEO KHÍA CẠNH (ABSA) CHO CÂU VÍ DỤ
Câu: The Laptop  is sleek and beautiful, and its  is great for travel. However, the cost is too steep and  slows down when gaming.
[2025-12-06 02:47:16] (2.4.2) Warning: reference sentiment does not exist or its number 0 is not equal to aspect number 1, text:  [B-ASP]Global Sentiment[E-ASP] The Laptop  is sleek and beautiful, and its  is great for travel. However, the cost i